# THIS NOTEBOOK HAS BEEN TESTED ONLY UNDER PYTHON 3.12
## It is advisable to use a venv to test this code

In [2]:
import sys
import os

def encontrar_carpeta(nombre_subruta, niveles_max=5):
    """
    Busca la carpeta 'nombre_subruta' empezando en la carpeta actual
    y subiendo hasta 'niveles_max' niveles hacia arriba. Devuelve la
    ruta absoluta si la encuentra; si no, lanza un error claro.
    Independiente del sistema operativo y de dónde arrancó el notebook.
    """
    base = os.getcwd()
    for _ in range(niveles_max + 1):
        candidata = os.path.join(base, *nombre_subruta)
        if os.path.isdir(candidata):
            return candidata
        base = os.path.dirname(base)  # sube un nivel
    raise FileNotFoundError(
        f"No encontré la carpeta {os.path.join(*nombre_subruta)} "
        f"partiendo de {os.getcwd()}"
    )

# Buscar la carpeta de los módulos sin asumir dónde arrancó el notebook
carpeta_preprocessing = encontrar_carpeta(("scripts", "preprocessing"))

# Agregarla a la ruta de búsqueda de Python
sys.path.insert(0, carpeta_preprocessing)

print("Carpeta encontrada:", carpeta_preprocessing)


Carpeta encontrada: e:\GIT_M2\codefest\base_de_conocimiento\scripts\preprocessing


In [3]:
## Descomentar si falla torch o numpy por version
#!pip uninstall torch torchvision torchaudio -y
#!pip cache purge

ruta_requirements = os.path.join(carpeta_preprocessing, "requirements.txt")

!{sys.executable} -m pip install -q -r "{ruta_requirements}"

In [4]:
#from pdf_preprocessor import process_pdf #deprecada usar text_processor
from text_processor import process_pdf
from pbf_preprocessor import process_pbf
from json_preprocessor import procesar_jsonl
from csv_preprocessor import process_csv_file
import tqdm, os

In [15]:
from tqdm import tqdm
import os
import statistics

carpeta_files = encontrar_carpeta(("src", "CORPUS CODEFEST AD ASTRA 2026"))

folders = [ls for ls in os.listdir(carpeta_files) if os.path.isdir(os.path.join(carpeta_files, ls))]

archivos_por_carpeta = {}

# Una barra de progreso por cada carpeta principal
for folder in folders:
    ruta_folder = os.path.join(carpeta_files, folder)

    # Primero recolectamos todos los archivos de esta carpeta
    todos = [
        os.path.join(root, file)
        for root, _, files in os.walk(ruta_folder)
        for file in files
    ]

    # Barra de progreso propia de esta carpeta, filtrando .DS_Store
    archivos = [
        f for f in tqdm(todos, desc=folder, unit="archivo")
        if os.path.basename(f) != ".DS_Store"
    ]

    archivos_por_carpeta[folder] = archivos

# Resumen final
print("\nResumen:")
for folder, archivos in archivos_por_carpeta.items():
    print(f"{folder}: {len(archivos)} archivos")

total = sum(len(archivos) for archivos in archivos_por_carpeta.values())
print(f"\nTotal: {total} archivos")


# ---------------------------------------------------------
# Distribución de tamaño por formato
# ---------------------------------------------------------

formatos = [".pdf", ".csv", ".json", ".pbf"]

print("\nDistribución de tamaños por formato:")

for formato in formatos:

    tamaños = [
        os.path.getsize(archivo) / (1024 * 1024)
        for archivos in archivos_por_carpeta.values()
        for archivo in archivos
        if os.path.splitext(archivo)[1].lower() == formato
    ]

    if tamaños:
        print(f"\n{formato.upper()}")
        print(f"Cantidad : {len(tamaños)}")
        print(f"Mínimo   : {min(tamaños):.2f} MB")
        print(f"Máximo   : {max(tamaños):.2f} MB")
        print(f"Media    : {statistics.mean(tamaños):.2f} MB")
        print(f"Mediana  : {statistics.median(tamaños):.2f} MB")
    else:
        print(f"\n{formato.upper()}")
        print("No hay archivos de este formato.")

# ---------------------------------------------------------
# 10 archivos más grandes por formato
# ---------------------------------------------------------

formatos = [".pdf", ".csv", ".json", ".pbf"]

print("\n10 archivos más grandes por formato:")

for formato in formatos:

    archivos_formato = [
        archivo
        for archivos in archivos_por_carpeta.values()
        for archivo in archivos
        if os.path.splitext(archivo)[1].lower() == formato
    ]

    # Ordenar de mayor a menor tamaño
    archivos_formato = sorted(
        archivos_formato,
        key=os.path.getsize,
        reverse=True
    )

    print(f"\n{'=' * 70}")
    print(f"TOP 10 {formato.upper()}")
    print(f"{'=' * 70}")

    for i, archivo in enumerate(archivos_formato[:100], start=1):
        tamaño_mb = os.path.getsize(archivo) / (1024 * 1024)

        print(
            f"{i:2}. {os.path.basename(archivo):50} "
            f"{tamaño_mb:10.2f} MB"
        )

F1_IA_y_Capacidades_Estrategicas: 100%|██████████| 462/462 [00:00<00:00, 390364.31archivo/s]


F3_Dinamicas_Territoriales: 100%|██████████| 902/902 [00:00<00:00, 205177.19archivo/s]


Resumen:
F1_IA_y_Capacidades_Estrategicas: 459 archivos
F2_Seguridad_Entorno_Espacial: 479 archivos
F3_Dinamicas_Territoriales: 899 archivos

Total: 1837 archivos

Distribución de tamaños por formato:

.PDF
Cantidad : 759
Mínimo   : 0.02 MB
Máximo   : 97.68 MB
Media    : 3.81 MB
Mediana  : 1.36 MB

.CSV
Cantidad : 26
Mínimo   : 0.00 MB
Máximo   : 33.30 MB
Media    : 3.02 MB
Mediana  : 0.02 MB

.JSON
Cantidad : 964
Mínimo   : 0.00 MB
Máximo   : 0.57 MB
Media    : 0.01 MB
Mediana  : 0.00 MB

.PBF
Cantidad : 73
Mínimo   : 0.00 MB
Máximo   : 1.00 MB
Media    : 0.22 MB
Mediana  : 0.22 MB

10 archivos más grandes por formato:

TOP 10 .PDF
 1. CSET_center-for-security-and-emerging-technology-4.pdf      97.68 MB
 2. RESDAL_atlas-2024-esp.pdf                               86.18 MB
 3. SWF_global-counterspace-capabilities-2026-hr.pdf        55.14 MB
 4. RESDAL_2014-complete2.pdf                               50.73 MB
 5. RESDAL_atlas-2016-ing-completo.pdf                      43.76 MB
 6. RESDA

In [6]:
from tqdm import tqdm

carpeta_files = encontrar_carpeta(("src", "CORPUS CODEFEST AD ASTRA 2026"))

folders = [ls for ls in os.listdir(carpeta_files) if os.path.isdir(os.path.join(carpeta_files, ls))]

archivos_por_carpeta = {}

# Una barra de progreso por cada carpeta principal
for folder in folders:
    ruta_folder = os.path.join(carpeta_files, folder)

    # Primero recolectamos todos los archivos de esta carpeta (para conocer el total)
    todos = [
        os.path.join(root, file)
        for root, _, files in os.walk(ruta_folder)
        for file in files
    ]

    # Barra de progreso propia de esta carpeta, filtrando .DS_Store
    archivos = [
        f for f in tqdm(todos, desc=folder, unit="archivo")
        if os.path.basename(f) != ".DS_Store"
    ]

    archivos_por_carpeta[folder] = archivos

# Resumen final
print("\nResumen:")
for folder, archivos in archivos_por_carpeta.items():
    print(f"{folder}: {len(archivos)} archivos")

total = sum(len(archivos) for archivos in archivos_por_carpeta.values())
print(f"\nTotal: {total} archivos")

F1_IA_y_Capacidades_Estrategicas: 100%|██████████| 462/462 [00:00<00:00, 220125.92archivo/s]


F3_Dinamicas_Territoriales: 100%|██████████| 902/902 [00:00<00:00, 271182.15archivo/s]


Resumen:
F1_IA_y_Capacidades_Estrategicas: 459 archivos
F2_Seguridad_Entorno_Espacial: 479 archivos
F3_Dinamicas_Territoriales: 899 archivos

Total: 1837 archivos


# Celda de ejecucion de todas las clases

In [7]:
### Escribe en disco por lotes para umentar la velocidad de procesamiento.

import os
import json
import queue
import datetime
import threading
from tqdm import tqdm

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
try:
    from huggingface_hub.utils import disable_progress_bars
    disable_progress_bars()
except Exception:
    pass

PROCESADORES = {
    ".pdf":  lambda file, fenomeno: process_pdf(pdf_path=file, fenomeno=fenomeno, imprimir_validacion=True),
    ".pbf":  lambda file, fenomeno: process_pbf(pbf_path=file, fenomeno=fenomeno),
    ".json": lambda file, fenomeno: procesar_jsonl(path=file, fenomeno=fenomeno),
    ".csv":  lambda file, fenomeno: process_csv_file(csv_path=file, fenomeno=fenomeno),
}

CARPETA_OUTPUT = os.path.join(os.path.dirname(encontrar_carpeta(["scripts"])), "output")
ARCHIVO_CORPUS    = os.path.join(CARPETA_OUTPUT, "metadata.json")     # grande -> gitignore
ARCHIVO_RESUMEN   = os.path.join(CARPETA_OUTPUT, "resumen.json")      # versionado
ARCHIVO_CHANGELOG = os.path.join(CARPETA_OUTPUT, "changelog.jsonl")   # log de ejecuciones

TAM_LOTE = 10
LIMITE_ARCHIVO_GRANDE = 10 * 1024 * 1024   # 10 MB
IN_PROGRESS, SUCCESS, FAILED = "IN PROGRESS", "SUCCESS", "FAILED"


def _ahora() -> str:
    return datetime.datetime.now().isoformat(timespec="seconds")


def _tamano_archivo(path: str) -> int:
    """Tamaño en bytes; 0 si el archivo no es accesible."""
    try:
        return os.path.getsize(path)
    except OSError:
        return 0


def _a_texto(chunks) -> str:
    """Normaliza la salida de un procesador a string, aunque devuelva lista o dict."""
    if not chunks:
        return ""
    if isinstance(chunks, str):
        return chunks
    if isinstance(chunks, (list, tuple)):
        return "\n".join(_a_texto(c) for c in chunks if c)
    try:
        return json.dumps(chunks, ensure_ascii=False)
    except (TypeError, ValueError):
        return str(chunks)


def _cargar_resumen_previo() -> dict:
    """Devuelve {path: n_chunks} de corridas anteriores. Solo contiene SUCCESS."""
    if not os.path.exists(ARCHIVO_RESUMEN):
        return {}
    try:
        with open(ARCHIVO_RESUMEN, "r", encoding="utf-8") as f:
            datos = json.load(f)
        procesados = datos.get("paths_procesados", {})
        if isinstance(procesados, list):          # compatibilidad con formato viejo
            return {p: None for p in procesados}
        return dict(procesados)
    except (json.JSONDecodeError, OSError):
        return {}


def _log_inmediato(ts, path, estado, error=None) -> None:
    """Escribe una línea del changelog y la fuerza a disco (sin buffer)."""
    registro = {"timestamp": ts, "path": path, "estado": estado}
    if error:
        registro["error"] = error
    with open(ARCHIVO_CHANGELOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())


class _Escritor(threading.Thread):
    """Escribe lotes (metadata + estado final + resumen) mientras el principal llena el siguiente."""

    def __init__(self, resumen_inicial: dict):
        super().__init__(daemon=True)
        self.cola = queue.Queue(maxsize=2)
        self.resumen = dict(resumen_inicial)
        self.error = None

    def encolar(self, lote):
        if lote:
            self.cola.put(lote)

    def cerrar(self):
        self.cola.put(None)
        self.join()

    def run(self):
        while True:
            lote = self.cola.get()
            if lote is None:
                break
            try:
                self._escribir_lote(lote)
            except Exception as e:
                self.error = e

    def _escribir_lote(self, lote):
        hubo_success = False
        with open(ARCHIVO_CORPUS, "a", encoding="utf-8") as f_meta, \
             open(ARCHIVO_CHANGELOG, "a", encoding="utf-8") as f_log:
            for reg in lote:
                # metadata (el IN PROGRESS ya se escribió antes de procesar)
                if reg["texto"]:
                    f_meta.write(reg["texto"] + "\n")
                # changelog: estado final
                final = {"timestamp": reg["ts_fin"], "path": reg["path"], "estado": reg["estado"]}
                if reg.get("error"):
                    final["error"] = reg["error"]
                f_log.write(json.dumps(final, ensure_ascii=False) + "\n")
                # resumen: solo SUCCESS
                if reg["estado"] == SUCCESS:
                    self.resumen[reg["path"]] = reg["n_chunks"]
                    hubo_success = True
            f_meta.flush()
            f_log.flush()
        if hubo_success:
            self._escribir_resumen()

    def _escribir_resumen(self):
        """Reescribe el resumen completo de forma atómica (tmp + replace)."""
        resumen = {
            "fecha": _ahora(),
            "archivos_procesados_total": len(self.resumen),
            "chunks_totales": sum(v for v in self.resumen.values() if isinstance(v, int)),
            "paths_procesados": dict(sorted(self.resumen.items())),
        }
        tmp = ARCHIVO_RESUMEN + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(resumen, f, ensure_ascii=False, indent=2)
        os.replace(tmp, ARCHIVO_RESUMEN)


def segmentar_archivos(modo_prueba: bool = False, continuar_segmentacion: bool = True) -> str:
    os.makedirs(CARPETA_OUTPUT, exist_ok=True)

    resumen_previo = _cargar_resumen_previo() if continuar_segmentacion else {}
    if not resumen_previo:                       # corrida desde cero -> truncar corpus
        open(ARCHIVO_CORPUS, "w", encoding="utf-8").close()

    escritor = _Escritor(resumen_previo)
    escritor.start()

    procesados = set(resumen_previo)
    formatos = set()
    lote, partes = [], []
    n_nuevos = n_chunks = 0

    try:
        for key in archivos_por_carpeta:
            fenomeno = key[1]
            for file in tqdm(archivos_por_carpeta[key], desc=str(key), unit="archivo"):
                if file in procesados:
                    continue

                extension = os.path.splitext(file)[1].lower()
                procesador = PROCESADORES.get(extension)
                if procesador is None:
                    continue
                if modo_prueba and extension in formatos:
                    continue

                # Archivos grandes se descargan solos, sin esperar a completar el lote
                es_grande = _tamano_archivo(file) > LIMITE_ARCHIVO_GRANDE
                if es_grande and lote:
                    escritor.encolar(lote)
                    lote = []

                # IN PROGRESS: se persiste ANTES de procesar (detección exacta)
                ts_inicio = _ahora()
                _log_inmediato(ts_inicio, file, IN_PROGRESS)

                try:
                    texto = _a_texto(procesador(file, fenomeno))
                    estado, err = SUCCESS, None
                except Exception as e:
                    texto, estado, err = "", FAILED, f"{type(e).__name__}: {e}"

                c = sum(1 for l in texto.splitlines() if l.strip())
                lote.append({
                    "path": file, "texto": texto, "n_chunks": c,
                    "estado": estado, "error": err, "ts_fin": _ahora(),
                })

                if estado == SUCCESS:
                    procesados.add(file)
                    formatos.add(extension)
                    partes.append(texto)
                    n_nuevos += 1
                    n_chunks += c

                # Buffer 1 para grandes; lote normal para el resto
                if es_grande or len(lote) >= TAM_LOTE:
                    escritor.encolar(lote)
                    lote = []

                if modo_prueba and len(formatos) == len(PROCESADORES):
                    break
            if modo_prueba and len(formatos) == len(PROCESADORES):
                break
    finally:
        escritor.encolar(lote)                   # último lote parcial
        escritor.cerrar()

    if escritor.error:
        print(f"Error del escritor: {escritor.error}")

    print(f"Nuevos: {n_nuevos} archivos, {n_chunks} chunks. Total acumulado: {len(procesados)}")
    return "\n".join(partes)


def filtrar_in_progress(marcar_failed: bool = False) -> list:
    """Devuelve las rutas cuyo último estado quedó IN PROGRESS (corrida interrumpida)."""
    if not os.path.exists(ARCHIVO_CHANGELOG):
        return []

    ultimo = {}
    with open(ARCHIVO_CHANGELOG, "r", encoding="utf-8") as f:
        for linea in f:
            try:
                reg = json.loads(linea)
                ultimo[reg["path"]] = reg["estado"]
            except (json.JSONDecodeError, KeyError):
                continue

    pendientes = [p for p, e in ultimo.items() if e == IN_PROGRESS]

    if marcar_failed and pendientes:
        with open(ARCHIVO_CHANGELOG, "a", encoding="utf-8") as f:
            for p in pendientes:
                f.write(json.dumps({
                    "timestamp": _ahora(), "path": p, "estado": FAILED,
                    "error": "Interrumpido: quedó IN PROGRESS",
                }, ensure_ascii=False) + "\n")

    return pendientes

## Ejecución con doble paralelismo

In [8]:
import os
import json
import queue
import datetime
import threading
import itertools
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from tqdm import tqdm

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # evita paralelismo anidado con los workers
try:
    from huggingface_hub.utils import disable_progress_bars
    disable_progress_bars()
except Exception:
    pass

PROCESADORES = {
    ".pdf":  lambda file, fenomeno: process_pdf(pdf_path=file, fenomeno=fenomeno, imprimir_validacion=False),
    ".pbf-skip":  lambda file, fenomeno: process_pbf(pbf_path=file, fenomeno=fenomeno),
    ".json": lambda file, fenomeno: procesar_jsonl(path=file, fenomeno=fenomeno),
    ".csv":  lambda file, fenomeno: process_csv_file(csv_path=file, fenomeno=fenomeno),
}

CARPETA_OUTPUT = os.path.join(os.path.dirname(encontrar_carpeta(["scripts"])), "output")
ARCHIVO_CORPUS    = os.path.join(CARPETA_OUTPUT, "metadata.json")
ARCHIVO_RESUMEN   = os.path.join(CARPETA_OUTPUT, "resumen.json")
ARCHIVO_CHANGELOG = os.path.join(CARPETA_OUTPUT, "changelog.jsonl")

TAM_LOTE = 10
LIMITE_ARCHIVO_GRANDE = 10 * 1024 * 1024   # 10 MB
IN_PROGRESS, SUCCESS, FAILED = "IN PROGRESS", "SUCCESS", "FAILED"

_LOCK_LOG = threading.Lock()   # el changelog lo escriben varios workers


def _ahora() -> str:
    return datetime.datetime.now().isoformat(timespec="seconds")


def _tamano_archivo(path: str) -> int:
    try:
        return os.path.getsize(path)
    except OSError:
        return 0


def _a_texto(chunks) -> str:
    """Normaliza la salida de un procesador a string, aunque devuelva lista o dict."""
    if not chunks:
        return ""
    if isinstance(chunks, str):
        return chunks
    if isinstance(chunks, (list, tuple)):
        return "\n".join(_a_texto(c) for c in chunks if c)
    try:
        return json.dumps(chunks, ensure_ascii=False)
    except (TypeError, ValueError):
        return str(chunks)


def _cargar_resumen_previo() -> dict:
    """Devuelve {path: n_chunks} de corridas anteriores. Solo contiene SUCCESS."""
    if not os.path.exists(ARCHIVO_RESUMEN):
        return {}
    try:
        with open(ARCHIVO_RESUMEN, "r", encoding="utf-8") as f:
            datos = json.load(f)
        procesados = datos.get("paths_procesados", {})
        if isinstance(procesados, list):
            return {p: None for p in procesados}
        return dict(procesados)
    except (json.JSONDecodeError, OSError):
        return {}


def _log_inmediato(ts, path, estado, error=None) -> None:
    """Escribe una línea del changelog y la fuerza a disco. Seguro entre hilos."""
    registro = {"timestamp": ts, "path": path, "estado": estado}
    if error:
        registro["error"] = error
    with _LOCK_LOG:
        with open(ARCHIVO_CHANGELOG, "a", encoding="utf-8") as f:
            f.write(json.dumps(registro, ensure_ascii=False) + "\n")
            f.flush()
            os.fsync(f.fileno())


def _procesar_uno(file, fenomeno, procesador) -> dict:
    """Se ejecuta en un worker. Registra IN PROGRESS y procesa el archivo."""
    ts_inicio = _ahora()
    _log_inmediato(ts_inicio, file, IN_PROGRESS)
    try:
        texto = _a_texto(procesador(file, fenomeno))
        estado, err = SUCCESS, None
    except Exception as e:
        texto, estado, err = "", FAILED, f"{type(e).__name__}: {e}"
    return {
        "path": file, "texto": texto, "estado": estado, "error": err,
        "n_chunks": sum(1 for l in texto.splitlines() if l.strip()),
        "es_grande": _tamano_archivo(file) > LIMITE_ARCHIVO_GRANDE,
        "ts_fin": _ahora(),
    }


class _Escritor(threading.Thread):
    """Escribe lotes (metadata + estado final + resumen) en paralelo al procesamiento."""

    def __init__(self, resumen_inicial: dict):
        super().__init__(daemon=True)
        self.cola = queue.Queue(maxsize=2)
        self.resumen = dict(resumen_inicial)
        self.error = None

    def encolar(self, lote):
        if lote:
            self.cola.put(lote)

    def cerrar(self):
        self.cola.put(None)
        self.join()

    def run(self):
        while True:
            lote = self.cola.get()
            if lote is None:
                break
            try:
                self._escribir_lote(lote)
            except Exception as e:
                self.error = e

    def _escribir_lote(self, lote):
        hubo_success = False
        with _LOCK_LOG:
            with open(ARCHIVO_CORPUS, "a", encoding="utf-8") as f_meta, \
                 open(ARCHIVO_CHANGELOG, "a", encoding="utf-8") as f_log:
                for reg in lote:
                    if reg["texto"]:
                        f_meta.write(reg["texto"] + "\n")
                    final = {"timestamp": reg["ts_fin"], "path": reg["path"], "estado": reg["estado"]}
                    if reg.get("error"):
                        final["error"] = reg["error"]
                    f_log.write(json.dumps(final, ensure_ascii=False) + "\n")
                    if reg["estado"] == SUCCESS:
                        self.resumen[reg["path"]] = reg["n_chunks"]
                        hubo_success = True
                f_meta.flush()
                f_log.flush()
        if hubo_success:
            self._escribir_resumen()

    def _escribir_resumen(self):
        resumen = {
            "fecha": _ahora(),
            "archivos_procesados_total": len(self.resumen),
            "chunks_totales": sum(v for v in self.resumen.values() if isinstance(v, int)),
            "paths_procesados": dict(sorted(self.resumen.items())),
        }
        tmp = ARCHIVO_RESUMEN + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(resumen, f, ensure_ascii=False, indent=2)
        os.replace(tmp, ARCHIVO_RESUMEN)


def _construir_tareas(procesados: set, modo_prueba: bool) -> list:
    """Arma la lista de (file, fenomeno, procesador) a ejecutar."""
    tareas, formatos = [], set()
    for key in archivos_por_carpeta:
        fenomeno = key[1]
        for file in archivos_por_carpeta[key]:
            if file in procesados:
                continue
            extension = os.path.splitext(file)[1].lower()
            procesador = PROCESADORES.get(extension)
            if procesador is None:
                continue
            if modo_prueba:
                if extension in formatos:
                    continue
                formatos.add(extension)
            tareas.append((file, fenomeno, procesador))
            if modo_prueba and len(formatos) == len(PROCESADORES):
                return tareas
    return tareas


def segmentar_archivos(modo_prueba: bool = False, continuar_segmentacion: bool = True,
                       max_workers: int = None) -> str:
    os.makedirs(CARPETA_OUTPUT, exist_ok=True)

    resumen_previo = _cargar_resumen_previo() if continuar_segmentacion else {}
    if not resumen_previo:
        open(ARCHIVO_CORPUS, "w", encoding="utf-8").close()

    if max_workers is None:
        max_workers = min(6, os.cpu_count() or 4)

    tareas = _construir_tareas(set(resumen_previo), modo_prueba)
    if not tareas:
        print("No hay archivos nuevos por procesar.")
        return ""

    escritor = _Escritor(resumen_previo)
    escritor.start()

    lote, partes = [], []
    n_ok = n_fail = n_chunks = 0
    ventana = max_workers * 2          # limita cuántos resultados viven en RAM
    tareas_iter = iter(tareas)

    barra = tqdm(total=len(tareas), desc="Procesando", unit="archivo")
    try:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futuros = {
                executor.submit(_procesar_uno, *t): t
                for t in itertools.islice(tareas_iter, ventana)
            }

            while futuros:
                hechos, _ = wait(futuros, return_when=FIRST_COMPLETED)
                for fut in hechos:
                    futuros.pop(fut)
                    reg = fut.result()

                    # archivo grande: descargar lo acumulado y mandarlo solo
                    if reg["es_grande"] and lote:
                        escritor.encolar(lote)
                        lote = []

                    lote.append(reg)
                    if reg["estado"] == SUCCESS:
                        partes.append(reg["texto"])
                        n_ok += 1
                        n_chunks += reg["n_chunks"]
                    else:
                        n_fail += 1

                    if reg["es_grande"] or len(lote) >= TAM_LOTE:
                        escritor.encolar(lote)
                        lote = []

                    barra.update(1)

                    # reponer la ventana con la siguiente tarea
                    siguiente = next(tareas_iter, None)
                    if siguiente is not None:
                        futuros[executor.submit(_procesar_uno, *siguiente)] = siguiente
    finally:
        barra.close()
        escritor.encolar(lote)
        escritor.cerrar()

    if escritor.error:
        print(f"Error del escritor: {escritor.error}")

    print(f"OK: {n_ok} | FAILED: {n_fail} | chunks nuevos: {n_chunks} | "
          f"total acumulado: {len(escritor.resumen)}")
    return "\n".join(partes)


def filtrar_in_progress(marcar_failed: bool = False) -> list:
    """Devuelve las rutas cuyo último estado quedó IN PROGRESS (corrida interrumpida)."""
    if not os.path.exists(ARCHIVO_CHANGELOG):
        return []

    ultimo = {}
    with open(ARCHIVO_CHANGELOG, "r", encoding="utf-8") as f:
        for linea in f:
            try:
                reg = json.loads(linea)
                ultimo[reg["path"]] = reg["estado"]
            except (json.JSONDecodeError, KeyError):
                continue

    pendientes = [p for p, e in ultimo.items() if e == IN_PROGRESS]

    if marcar_failed and pendientes:
        with open(ARCHIVO_CHANGELOG, "a", encoding="utf-8") as f:
            for p in pendientes:
                f.write(json.dumps({
                    "timestamp": _ahora(), "path": p, "estado": FAILED,
                    "error": "Interrumpido: quedó IN PROGRESS",
                }, ensure_ascii=False) + "\n")

    return pendientes

In [9]:
metadata = segmentar_archivos(modo_prueba=False,continuar_segmentacion=True) 

Procesando:  67%|██████▋   | 467/696 [36:01<20:08,  5.28s/archivo][transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (772 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (934 > 512). Running this sequence through the model will result in indexing errors
Procesando:  90%|█████████ | 628/696 [1:15:02<03:28,  3.06s/archivo][transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (567 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (565 > 512). Running this sequence through the model will result in indexing errors
Procesando: 100%|██████████| 696/696 [1:38:28<00:00,  8.49s/archivo] 


OK: 689 | FAILED: 7 | chunks nuevos: 268196 | total acumulado: 1742


In [11]:
def validar_resumen_vs_metadata(verbose: bool = True) -> dict:
    """
    Verifica la consistencia entre resumen.json y metadata.json.

    Devuelve un dict con el resultado. Comprueba:
      - Que ambos archivos existan.
      - Que el resumen sea un JSON válido con 'paths_procesados'.
      - Que cada path listado en el resumen exista en disco.
      - Que la suma de chunks del resumen coincida con las líneas no vacías del corpus.
    """
    reporte = {
        "ok": False,
        "existe_resumen": False,
        "existe_metadata": False,
        "paths_en_resumen": 0,
        "paths_inexistentes_en_disco": [],
        "chunks_declarados": 0,
        "lineas_en_metadata": 0,
        "coincide_conteo": False,
        "errores": [],
    }

    # 1. Existencia de archivos
    reporte["existe_resumen"] = os.path.exists(ARCHIVO_RESUMEN)
    reporte["existe_metadata"] = os.path.exists(ARCHIVO_CORPUS)

    if not reporte["existe_resumen"]:
        reporte["errores"].append("No existe resumen.json")
    if not reporte["existe_metadata"]:
        reporte["errores"].append("No existe metadata.json")
    if reporte["errores"]:
        if verbose:
            _imprimir_reporte(reporte)
        return reporte

    # 2. Leer el resumen
    try:
        with open(ARCHIVO_RESUMEN, "r", encoding="utf-8") as f:
            resumen = json.load(f)
    except (json.JSONDecodeError, OSError) as e:
        reporte["errores"].append(f"No se pudo leer resumen.json: {e}")
        if verbose:
            _imprimir_reporte(reporte)
        return reporte

    paths = resumen.get("paths_procesados", {})
    if isinstance(paths, list):                 # compatibilidad con formato viejo
        paths = {p: None for p in paths}

    reporte["paths_en_resumen"] = len(paths)

    # 3. Cada path del resumen debe existir físicamente en disco
    reporte["paths_inexistentes_en_disco"] = [p for p in paths if not os.path.exists(p)]

    # 4. Suma de chunks declarados (solo los que tienen conteo numérico)
    reporte["chunks_declarados"] = sum(v for v in paths.values() if isinstance(v, int))

    # 5. Contar líneas no vacías reales en el corpus
    try:
        with open(ARCHIVO_CORPUS, "r", encoding="utf-8") as f:
            reporte["lineas_en_metadata"] = sum(1 for linea in f if linea.strip())
    except OSError as e:
        reporte["errores"].append(f"No se pudo leer metadata.json: {e}")
        if verbose:
            _imprimir_reporte(reporte)
        return reporte

    # 6. Comparar conteos
    reporte["coincide_conteo"] = (
        reporte["chunks_declarados"] == reporte["lineas_en_metadata"]
    )

    if reporte["paths_inexistentes_en_disco"]:
        reporte["errores"].append(
            f"{len(reporte['paths_inexistentes_en_disco'])} path(s) del resumen no existen en disco"
        )
    if not reporte["coincide_conteo"]:
        reporte["errores"].append(
            f"Desajuste de chunks: resumen declara {reporte['chunks_declarados']}, "
            f"metadata tiene {reporte['lineas_en_metadata']} líneas"
        )

    reporte["ok"] = not reporte["errores"]

    if verbose:
        _imprimir_reporte(reporte)
    return reporte


def _imprimir_reporte(reporte: dict) -> None:
    estado = "✔ CONSISTENTE" if reporte["ok"] else "�’ INCONSISTENTE"
    print(f"\n=== Validación resumen vs metadata: {estado} ===")
    print(f"  Paths en resumen:        {reporte['paths_en_resumen']}")
    print(f"  Chunks declarados:       {reporte['chunks_declarados']}")
    print(f"  Líneas en metadata:      {reporte['lineas_en_metadata']}")
    print(f"  Conteo coincide:         {reporte['coincide_conteo']}")
    if reporte["paths_inexistentes_en_disco"]:
        print(f"  Paths que no existen en disco ({len(reporte['paths_inexistentes_en_disco'])}):")
        for p in reporte["paths_inexistentes_en_disco"][:10]:
            print(f"    - {p}")
        if len(reporte["paths_inexistentes_en_disco"]) > 10:
            print(f"    ... y {len(reporte['paths_inexistentes_en_disco']) - 10} más")
    if reporte["errores"]:
        print("  Errores:")
        for e in reporte["errores"]:
            print(f"    - {e}")
    print()

In [12]:
validar_resumen_vs_metadata(True)


=== Validación resumen vs metadata: �’ INCONSISTENTE ===
  Paths en resumen:        1742
  Chunks declarados:       456776
  Líneas en metadata:      268190
  Conteo coincide:         False
  Errores:
    - Desajuste de chunks: resumen declara 456776, metadata tiene 268190 líneas



{'ok': False,
 'existe_resumen': True,
 'existe_metadata': True,
 'paths_en_resumen': 1742,
 'paths_inexistentes_en_disco': [],
 'chunks_declarados': 456776,
 'lineas_en_metadata': 268190,
 'coincide_conteo': False,
 'errores': ['Desajuste de chunks: resumen declara 456776, metadata tiene 268190 líneas']}

In [13]:
def diagnosticar_faltantes(verbose: bool = True) -> dict:
    """
    Con el formato actual solo puede reportar faltantes a nivel de ARCHIVO, no de chunk.
    Cruza resumen (SUCCESS) contra el changelog para hallar archivos incompletos.
    """
    reporte = {"faltante_total_chunks": 0, "archivos_no_success": [], "paths_sin_disco": []}

    # Estado final de cada path según el changelog
    estados = {}
    if os.path.exists(ARCHIVO_CHANGELOG):
        with open(ARCHIVO_CHANGELOG, "r", encoding="utf-8") as f:
            for linea in f:
                try:
                    reg = json.loads(linea)
                    estados[reg["path"]] = reg["estado"]
                except (json.JSONDecodeError, KeyError):
                    continue

    # Paths que se intentaron pero no terminaron en SUCCESS
    reporte["archivos_no_success"] = [
        p for p, e in estados.items() if e != SUCCESS
    ]

    # Desajuste global de conteo (resumen vs líneas reales)
    resumen_paths = {}
    if os.path.exists(ARCHIVO_RESUMEN):
        with open(ARCHIVO_RESUMEN, "r", encoding="utf-8") as f:
            datos = json.load(f)
        resumen_paths = datos.get("paths_procesados", {})
        if isinstance(resumen_paths, list):
            resumen_paths = {p: None for p in resumen_paths}

    declarados = sum(v for v in resumen_paths.values() if isinstance(v, int))
    lineas = 0
    if os.path.exists(ARCHIVO_CORPUS):
        with open(ARCHIVO_CORPUS, "r", encoding="utf-8") as f:
            lineas = sum(1 for l in f if l.strip())
    reporte["faltante_total_chunks"] = declarados - lineas

    reporte["paths_sin_disco"] = [p for p in resumen_paths if not os.path.exists(p)]

    if verbose:
        print(f"Chunks declarados vs escritos: faltan {reporte['faltante_total_chunks']} en total")
        print(f"Archivos que no llegaron a SUCCESS: {len(reporte['archivos_no_success'])}")
        for p in reporte["archivos_no_success"][:10]:
            print(f"   - [{estados[p]}] {p}")
    return reporte 

In [14]:
diagnosticar_faltantes(True)

Chunks declarados vs escritos: faltan 188586 en total
Archivos que no llegaron a SUCCESS: 13
   - [IN PROGRESS] e:\GIT_M2\codefest\base_de_conocimiento\src\CORPUS CODEFEST AD ASTRA 2026\F1_IA_y_Capacidades_Estrategicas\AI_Index_Stanford\pdfs\AIINDEX_ai-index-report-2017.pdf
   - [IN PROGRESS] e:\GIT_M2\codefest\base_de_conocimiento\src\CORPUS CODEFEST AD ASTRA 2026\F1_IA_y_Capacidades_Estrategicas\AI_Index_Stanford\pdfs\AIINDEX_ai-index-report-2018.pdf
   - [IN PROGRESS] e:\GIT_M2\codefest\base_de_conocimiento\src\CORPUS CODEFEST AD ASTRA 2026\F1_IA_y_Capacidades_Estrategicas\AI_Index_Stanford\pdfs\AIINDEX_ai-index-report-2019.pdf
   - [IN PROGRESS] e:\GIT_M2\codefest\base_de_conocimiento\src\CORPUS CODEFEST AD ASTRA 2026\F1_IA_y_Capacidades_Estrategicas\AI_Index_Stanford\pdfs\AIINDEX_ai-index-report-2021.pdf
   - [IN PROGRESS] e:\GIT_M2\codefest\base_de_conocimiento\src\CORPUS CODEFEST AD ASTRA 2026\F1_IA_y_Capacidades_Estrategicas\AI_Index_Stanford\pdfs\AIINDEX_ai-index-report-2022.p

{'faltante_total_chunks': 188586,
 'archivos_no_success': ['e:\\GIT_M2\\codefest\\base_de_conocimiento\\src\\CORPUS CODEFEST AD ASTRA 2026\\F1_IA_y_Capacidades_Estrategicas\\AI_Index_Stanford\\pdfs\\AIINDEX_ai-index-report-2017.pdf',
  'e:\\GIT_M2\\codefest\\base_de_conocimiento\\src\\CORPUS CODEFEST AD ASTRA 2026\\F1_IA_y_Capacidades_Estrategicas\\AI_Index_Stanford\\pdfs\\AIINDEX_ai-index-report-2018.pdf',
  'e:\\GIT_M2\\codefest\\base_de_conocimiento\\src\\CORPUS CODEFEST AD ASTRA 2026\\F1_IA_y_Capacidades_Estrategicas\\AI_Index_Stanford\\pdfs\\AIINDEX_ai-index-report-2019.pdf',
  'e:\\GIT_M2\\codefest\\base_de_conocimiento\\src\\CORPUS CODEFEST AD ASTRA 2026\\F1_IA_y_Capacidades_Estrategicas\\AI_Index_Stanford\\pdfs\\AIINDEX_ai-index-report-2021.pdf',
  'e:\\GIT_M2\\codefest\\base_de_conocimiento\\src\\CORPUS CODEFEST AD ASTRA 2026\\F1_IA_y_Capacidades_Estrategicas\\AI_Index_Stanford\\pdfs\\AIINDEX_ai-index-report-2022.pdf',
  'e:\\GIT_M2\\codefest\\base_de_conocimiento\\src\\CORPUS

In [10]:
import 7656 

SyntaxError: invalid syntax (1176506117.py, line 1)

In [ ]:
#pip install mapbox-vector-tile

In [ ]:
"""import mapbox_vector_tile

with open(r"C:\git local\codefest\base_de_conocimiento\src\CORPUS CODEFEST AD ASTRA 2026\F3_Dinamicas_Territoriales\Amazon_Underworld\tiles\3\2\AMAZONUW_3.pbf", "rb") as f:
    data = f.read()

tile = mapbox_vector_tile.decode(data)

# Ver las capas disponibles
for layer_name, layer in tile.items():
    print(f"Capa: {layer_name} -> {len(layer['features'])} features")

# Inspeccionar features de una capa
for layer_name, layer in tile.items():
    for feature in layer["features"][:5]:
        print(feature["geometry"], feature["properties"])"""

In [ ]:
#pip install --user -q mapbox-vector-tile shapely matplotlib

In [ ]:
"""import gzip
import mapbox_vector_tile
import matplotlib.pyplot as plt
from shapely.geometry import shape

ruta = r"C:\git local\codefest\base_de_conocimiento\src\CORPUS CODEFEST AD ASTRA 2026\F3_Dinamicas_Territoriales\Amazon_Underworld\tiles\4\4\AMAZONUW_8.pbf"

with open(ruta, "rb") as f:
    data = f.read()

# Descomprime si viene en gzip
if data[:2] == b"\x1f\x8b":
    data = gzip.decompress(data)

tile = mapbox_vector_tile.decode(data)

fig, ax = plt.subplots(figsize=(10, 10))

for layer_name, layer in tile.items():
    for feature in layer["features"]:
        geom = shape(feature["geometry"])
        gtype = geom.geom_type

        if gtype == "Point":
            ax.plot(geom.x, geom.y, "o", markersize=2)
        elif gtype == "MultiPoint":
            for p in geom.geoms:
                ax.plot(p.x, p.y, "o", markersize=2)
        elif gtype == "LineString":
            x, y = geom.xy
            ax.plot(x, y, linewidth=0.5)
        elif gtype == "MultiLineString":
            for line in geom.geoms:
                x, y = line.xy
                ax.plot(x, y, linewidth=0.5)
        elif gtype == "Polygon":
            x, y = geom.exterior.xy
            ax.fill(x, y, alpha=0.4)
        elif gtype == "MultiPolygon":
            for poly in geom.geoms:
                x, y = poly.exterior.xy
                ax.fill(x, y, alpha=0.4)

ax.set_aspect("equal")
ax.axis("off")
plt.savefig("tile.png", dpi=150, bbox_inches="tight")
plt.show()

print("Imagen guardada como tile.png")"""

In [ ]:
#chunks=process_pbf(fenomeno=3, fuente="AMAZONUW_3.pbf", pbf_path=r"C:\git local\codefest\base_de_conocimiento\src\CORPUS CODEFEST AD ASTRA 2026\F3_Dinamicas_Territoriales\Amazon_Underworld\tiles\3\2\AMAZONUW_3.pbf")

In [ ]:
"""import json

# chunks puede venir como string JSONL (una linea JSON por chunk) o ya como lista de dicts
if isinstance(chunks, str):
    registros = [json.loads(linea) for linea in chunks.splitlines() if linea.strip()]
else:
    registros = chunks

print(json.dumps(registros, indent=2, ensure_ascii=False))"""